In [ ]:
# =============================================================
# NOTEBOOK 02 — FEATURE ENGINEERING
# En MLOps, Feature Engineering es el proceso de crear nuevas variables
# a partir de las existentes para que el modelo aprenda mejor.
# Un buen feature puede mejorar el modelo más que cambiar el algoritmo.
# Este notebook representa el trabajo del Data Scientist/MLE
# ANTES de automatizar el proceso en un script de producción (engineer.py).
# =============================================================

# ---
# 🧪 Imports
import pandas as pd              # Para manipular el dataset como tabla (DataFrame)
import numpy as np               # Para operaciones numéricas (ej: manejar infinitos con np.inf)
from datetime import datetime    # Para obtener el año actual y calcular la edad de las casas
import seaborn as sns            # Para visualizaciones estadísticas
import matplotlib.pyplot as plt  # Para crear y mostrar gráficos

# ---
# 📥 Load Cleaned Data
# En MLOps, cargamos desde data/processed/ — datos ya limpios del paso anterior (00_data_engineering).
# Nunca volvemos a cargar desde raw/ en este punto del pipeline.
df = pd.read_csv("../data/processed/cleaned_house_data.csv")  # Lee el CSV limpio como DataFrame
df.head()  # Muestra las primeras 5 filas para verificar que los datos se cargaron correctamente


In [ ]:
# ---
# 🔍 3. Initial Data Exploration
# Aunque ya limpiamos los datos en el notebook 00, siempre es buena práctica
# hacer una revisión rápida al inicio de cada notebook para confirmar el estado de los datos.

# Print dataset shape: rows × columns
print("Dataset shape (colums, rows):", df.shape)  # .shape devuelve (n_filas, n_columnas)
                                                    # Nota: el mensaje dice 'colums, rows' pero .shape devuelve (rows, cols)

# Print summary of column names, types, null counts, etc.
df.info()  # Muestra: nombre de columna, tipo de dato (int64, float64, object) y valores no-nulos
           # Útil para verificar que no se introdujeron NaN en pasos anteriores del pipeline

# Show basic stats (mean, min, max, std) for numeric columns
df.describe()  # Estadísticas descriptivas: count, mean, std, min, 25%, 50%, 75%, max
               # Sirve para detectar outliers (ej: max muy alejado del 75%)

# ---
# 🧠 Why this matters:
# This gives a quick overview of the data:
#  Are there missing values?       → Si hay NaN, el Feature Engineering puede fallar
#  What kind of columns are there (numeric, text)?  → Determina qué transformaciones aplicar
#  Do the values look reasonable?  → Detecta si algo salió mal en el paso de limpieza


In [ ]:
# ---
# 🎯 Feature Engineering (Human-Readable)
# En MLOps, creamos features con conocimiento del dominio (domain knowledge).
# No es solo matemática — es entender QUÉ variables son relevantes para predecir el precio.

# Feature 1: Age of the house
# Hipótesis: las casas más viejas tienden a valer menos (o más, si son históricas)
df['house_age'] = datetime.now().year - df['year_built']  # datetime.now().year: obtiene el año actual (ej: 2026)
                                                           # Resta el año de construcción para obtener la edad en años
                                                           # Crea una nueva columna 'house_age' en el DataFrame

# Feature 2: Price per square foot
# Hipótesis: normalizar el precio por tamaño permite comparar casas de distintos tamaños
df['price_per_sqft'] = df['price'] / df['sqft']  # Divide precio entre metros cuadrados
                                                   # Una casa de $300k con 1000 sqft tiene price_per_sqft=300
                                                   # Una casa de $300k con 3000 sqft tiene price_per_sqft=100

# Feature 3: Ratio of bedrooms to bathrooms
# Hipótesis: el balance entre habitaciones y baños afecta la comodidad y por ende el precio
df['bed_bath_ratio'] = df['bedrooms'] / df['bathrooms']  # Divide número de habitaciones entre baños
                                                          # Ej: 3 habitaciones / 2 baños = ratio 1.5

# Handle division by zero and infinite values
# Si bathrooms=0, la división genera inf (infinito) o NaN — debemos manejarlo
df['bed_bath_ratio'] = df['bed_bath_ratio'].replace([np.inf, -np.inf], np.nan)  # Reemplaza infinitos con NaN
df['bed_bath_ratio'] = df['bed_bath_ratio'].fillna(0)  # Reemplaza NaN con 0 (casa sin baños → ratio 0)
                                                        # Esto se puede encadenar: .replace(...).fillna(0)

# 📊 Quick Look at Engineered Data
df[['house_age', 'price_per_sqft', 'bed_bath_ratio']].describe()  # Muestra estadísticas solo de las 3 nuevas features
                                                                    # Para verificar que los valores son razonables


In [ ]:
# 📊 5. Visualize the New Features
# En MLOps, siempre visualizamos las nuevas features para verificar que tienen sentido.
# Una feature con distribución extraña (ej: todos los valores iguales) no aporta información al modelo.

# Plot distributions of the new features to check if they make sense

fig, axs = plt.subplots(1, 3, figsize=(18, 5))  # Crea una figura con 1 fila y 3 columnas de subgráficos
                                                  # figsize=(18, 5): 18 pulgadas de ancho, 5 de alto
                                                  # axs: array con los 3 ejes (subgráficos)

sns.histplot(df['house_age'], kde=True, ax=axs[0])  # Histograma de 'house_age' en el primer subgráfico (axs[0])
                                                      # kde=True: curva de densidad suavizada
axs[0].set_title('Distribution of House Age')        # Título del primer subgráfico

sns.histplot(df['price_per_sqft'], kde=True, ax=axs[1])  # Histograma de 'price_per_sqft' en el segundo subgráfico
axs[1].set_title('Price per Sqft')                        # Título del segundo subgráfico

sns.histplot(df['bed_bath_ratio'], kde=True, ax=axs[2])  # Histograma de 'bed_bath_ratio' en el tercer subgráfico
axs[2].set_title('Bedroom to Bathroom Ratio')             # Título del tercer subgráfico

plt.tight_layout()  # Ajusta automáticamente el espaciado entre subgráficos para que no se solapan
plt.show()          # Renderiza y muestra los 3 gráficos


In [ ]:
# 💾 Save Final Engineered Features (Human-Readable CSV)
# En MLOps, guardamos el dataset con las nuevas features en data/processed/
# para que el siguiente paso del pipeline (03_experimentation) lo use directamente.
# Esto hace el pipeline modular: cada notebook/script tiene una responsabilidad clara.

selected_columns = [                    # Lista de columnas que queremos conservar en el archivo final
    'price',                            # Target variable (lo que queremos predecir)
    'sqft',                             # Feature original: metros cuadrados
    'bedrooms',                         # Feature original: número de habitaciones
    'bathrooms',                        # Feature original: número de baños
    'location',                         # Feature original: ubicación (categórica)
    'year_built',                       # Feature original: año de construcción
    'condition',                        # Feature original: condición de la casa
    'house_age',                        # Feature nueva: edad de la casa (creada arriba)
    'price_per_sqft',                   # Feature nueva: precio por pie cuadrado (creada arriba)
    'bed_bath_ratio'                    # Feature nueva: ratio habitaciones/baños (creada arriba)
]

df[selected_columns].to_csv("../data/processed/data_scientists_features.csv", index=False)
# df[selected_columns]: selecciona solo las columnas de la lista (descarta el resto)
# .to_csv(): guarda el DataFrame como archivo CSV en la ruta especificada
# index=False: no incluye el índice numérico de pandas como columna extra en el CSV

print("✅ Saved human-readable engineered features to ../data/processed/data_scientists_features.csv")


In [ ]:
# ✅ 8. Summary
# Resumen de lo que hicimos en este notebook:

# - Creamos features con conocimiento del dominio:
#   house_age: edad de la casa
#   price_per_sqft: precio normalizado por tamaño
#   bed_bath_ratio: balance entre habitaciones y baños

# - Exploramos y visualizamos las nuevas features para validar que tienen sentido

# - Guardamos el dataset enriquecido en data/processed/ para el siguiente paso del pipeline

# 👨‍🔬 Este notebook representa el workflow del Data Scientist/MLE para Feature Engineering.
# En producción (MLOps), estos pasos se automatizan en un script como 'engineer.py'
# que corre como parte de un pipeline orquestado (ej: con Airflow, Prefect, o GitHub Actions).

# 📦 Siguiente paso: Notebook 03 — Experimentación para encontrar el mejor algoritmo
# y sus configuraciones óptimas (hiperparámetros).
